# Procedural Graphs: a walkthrough in code

*Giving an LLM agent an explicit, editable map of "how this kind of task is done" — injecting the relevant corner of that map into its prompt at every step, and letting a second LLM redraw the map from the agent's own successes and failures.*

Agents get better at a task the way people do: by accumulating procedure. *Search for the specific entity, not the whole question. Raise money before the surge, not during it. Don't resubmit while a request is pending.* Today that knowledge lives in one of two places — a system prompt that grows until nobody dares edit it, or model weights you can't inspect. Neither is something you can diff, review, or roll back.

**Procedural Graphs** ([arXiv 2609.09153](https://arxiv.org/abs/2609.09153)) proposes a third home: a small graph whose nodes are procedures (tool calls, reasoning steps, task states) and whose edges say *when* to move from one to the next, *how*, and *what to avoid*. The method has two halves:

- **Online guidance** — at every step, find where the agent is in the graph, cut out the neighborhood around that spot, and have an LLM turn it into two or three sentences of advice for the very next move.
- **Offline self-evolution** — run the agent on a batch of tasks, show a stronger "refiner" LLM the best and worst attempts, let it propose edits to the graph, and keep the edits only if a held-out validation set says they helped.

This repo is a deliberately small reference implementation of both halves on LangGraph — about 2,000 lines, no framework. This notebook is the walkthrough: every mechanism is exercised by a cell, on the HotpotQA scenario, and everything you see is produced live rather than quoted. Sections 1–9 and 11 need no API key; sections 10 and 12 make real LLM calls (about $1.50 in total, mostly section 12, which runs a real three-round evolution).

Setup: `uv sync`, `cp .env.example .env` (set `OPENROUTER_API_KEY`), `uv run pg gen-data hotpotqa`, then `uv run --with jupyterlab jupyter lab` from the repo root.

## The conceptual map: who does what, and who checks whom

Before any code, here is the whole method on one page. Every later section is one row of this table seen up close.

| Role | What it is | What it sees | What it produces | Who checks it |
|---|---|---|---|---|
| **Solver** | the agent being improved (a small LLM with tools) | task, its own history, this turn's guidance | tool calls | the environment's score |
| **Guidance model** | the same small LLM, called once per step | task, last 3 steps, the graph neighborhood around the last tool call | 2–5 sentences of advice, discarded after one turn | nothing directly — only through the solver's score |
| **Graph** | a JSON file: procedures as nodes; condition / guidance / pitfalls on edges | – | the only thing that persists and improves | the validation gate, then a person reading it |
| **Refiner** | a stronger LLM, called once per round | environment description, tools, the whole graph, the 3 best and 3 worst trajectories, rejection memory | a typed `EditSet` (add/delete nodes and edges) plus a rationale | the validation gate |
| **Validation gate** | plain code, no LLM | the candidate graph's score on a fixed held-out split | accept (score ≥ current) or reject → rejection memory | the untouched test split, with controls |

**How a graph is developed.** It starts as a *skeleton* — `Start`, `End`, one node per tool, no edges — or as a hand-written expert graph. Then, per round: (1) run the agent on a train batch **with the current graph**; (2) rank the trajectories and hand the refiner the best and worst, because contrast is the signal; (3) the refiner returns edits as data, never a rewritten graph; (4) the edits are applied to a **copy**; (5) the copy is run on the validation split with the full online machinery; (6) it replaces the current graph only if it scores at least as well. A rejected proposal is summarised into the refiner's next prompt so it is not tried again.

**How it is maintained.** The graph is an ordinary versioned file. Every round writes the graph it kept, the edits it proposed (accepted or not), the refiner's exact prompt, its rationale, and every trajectory — so any edge can be traced to the round, the evidence and the score that put it there, and any round can be rolled back by pointing `--init` at an earlier file. You can also edit the JSON by hand and evaluate your version against the original.

**Supervision — and how the supervisor is evaluated.** There is no human in the loop and no LLM judge. The refiner is the only thing that "supervises" the agent, and the design's central move is that **the refiner is never trusted**: its proposals are hypotheses, and the only test is behavioural — does an agent guided by the edited graph score better on tasks the refiner never saw? That is the whole evaluation of the supervisor, and it is per proposal. Three consequences worth knowing up front:

- *The rationale is never verified.* A proposal can be accepted for reasons unrelated to the story the refiner tells about it. The rationale is documentation for the human reader, not evidence.
- *The gate is itself noisy.* One validation pass is one sample of a stochastic agent, and "keep the best of several candidates" biases the kept score upward. So validation scores cannot be the final word; the gate is in turn checked by a **test split the loop never touches**, compared pairwise against no graph, against the edgeless skeleton (which isolates "the graph's content helps" from "an extra LLM call per step helps"), and against the expert graph.
- *The guidance model is never evaluated alone.* It is judged only through the solver's outcomes, which is why the skeleton control matters.

Human oversight therefore lives *outside* the loop, in the artifacts: the per-round diff and timeline (`analysis/graph_evolution.py`), the saved refiner prompts, and the paired test-split statistics (`analysis/paired_comparison.py`).

The rest of this notebook takes these in mechanical order: the graph data structure, the environment and the solver loop, the guidance loop that injects the graph, the evolution loop that edits it, and how to read what it produced. Whether any of it helped on the paper's finance benchmark is in [`walkthrough.md`](walkthrough.md), because that experiment costs a day and $70.

## 1. The map of the repo

```
src/pg/
  graph.py        the data structure: Node / Edge / EditSet / ProceduralGraph (neighborhood, serialize, apply)
  guidance.py     ONLINE half: localize -> neighborhood -> guidance LLM
  agent.py        the solver loop (LangGraph StateGraph); where guidance enters the prompt
  refiner.py      OFFLINE half, part 1: the refiner prompt and the structured EditSet reply
  evolve.py       OFFLINE half, part 2: rollout -> refine -> validate -> accept/reject + rejection memory
  evaluate.py     run a split under several graphs, side by side
  trajectory.py   Step / Trajectory records, summaries, per-episode rows
  llm.py          the only place an LLM client is built (OpenRouter), plus token/cost accounting
  tracking.py     optional Weights & Biases logging
  cli.py          `pg gen-data | evolve | eval`
  envs/
    base.py             the ONLY interface a scenario implements: Environment + Episode
    hotpotqa/           multi-hop QA: short episodes, cheap to iterate on
    finance/            synthetic 24-month startup simulator
    enterprisearena/    adapter over the EnterpriseArena CFO benchmark (132-month episodes)
graphs/           hand-written "expert" graphs; evolved graphs land in graphs/evolved/
analysis/         not needed to run anything: paired statistics over eval runs, graph-evolution diff / Mermaid /
                  timeline, this walkthrough
```

The rule that keeps this readable: **nothing outside `envs/` knows what task it is solving.** `graph`, `guidance`, `agent`, `refiner` and `evolve` only ever see tool names, text observations and a score.

Here is how the pieces connect:

```mermaid
flowchart LR
  subgraph online["Online: every step of every episode"]
    direction LR
    S[last tool call] --> M["match()<br/>guidance.py"]
    M --> N["neighborhood(h=2)<br/>graph.py"]
    N --> G["guidance LLM<br/>guidance.py"]
    G --> P["solver system prompt<br/>agent.py"]
    P --> T[tool call] --> S
  end
  subgraph offline["Offline: once per evolution round"]
    direction LR
    R["rollout on train batch<br/>agent.run_batch"] --> K[best + worst trajectories]
    K --> F["refiner LLM -> EditSet<br/>refiner.py"]
    F --> A["graph.apply(edits)"]
    A --> V{"val score >= current?<br/>evolve.py"}
    V -- yes --> ACC[accept: new current graph]
    V -- no --> REJ[rejection memory] --> F
  end
  ACC -.-> N
```

## 2. The data structure: a procedural graph

Open the hand-written HotpotQA graph. Three node types, four relation types, and three pieces of free text on every edge: **condition** (when the transition applies), **guidance** (how), **pitfalls** (what to avoid). That is the whole knowledge representation.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()))  # so `analysis.*` imports work from either directory

from pg.config import ROOT, Config
from pg.graph import ProceduralGraph, Node, Edge, NodeType, Relation
from analysis.graph_evolution import graph_to_mermaid
from IPython.display import Markdown, display

def show(graph, active=None):
    """Draw a graph as Mermaid: rounded = STATE, box = ACTION, hexagon = REASONING; green = the active node."""
    display(Markdown("```mermaid\n" + graph_to_mermaid(graph, active) + "\n```"))

expert = ProceduralGraph.load(ROOT / "graphs" / "hotpotqa_expert.json")
print(expert.summary())
for n in expert.nodes:
    print(f"  [{n.id}] ({n.type.value}) {n.description}")
e = expert.edges[3]
print("\none edge:", e.model_dump_json(indent=2))

7 nodes, 10 edges
  [Start] (STATE) A multi-hop question has been posed.
  [search] (ACTION) Keyword search over the context paragraphs.
  [lookup] (ACTION) Read a full paragraph by title.
  [analyze] (REASONING) Extract the bridge entity or the facts found so far and decide what the second hop needs.
  [verify] (REASONING) Check that both hops are supported by retrieved text and that the answer has the requested form.
  [finish] (ACTION) Submit a short answer span.
  [End] (STATE) Answer submitted.

one edge: {
  "src": "analyze",
  "rel": "LEADS_TO",
  "dst": "search",
  "condition": "A bridge entity or second comparison entity is identified but its paragraph has not been retrieved.",
  "guidance": "Second hop: search for the bridge entity by name together with the attribute the question asks about.",
  "pitfalls": "Do not stop after one hop; most HotpotQA questions need two paragraphs."
}


The schema is pydantic, so the same classes are the JSON file format, the refiner's structured-output schema and the in-memory structure. The `Field` descriptions are literally read by the refiner LLM. One of them carries the convention the whole method rests on:

In [2]:
for name, f in Node.model_fields.items():
    print(f"Node.{name}: {f.description or '(no description)'}")
print([r.value for r in Relation])

Node.id: Unique node id. For ACTION nodes this must equal the tool name.
Node.type: (no description)
Node.description: (no description)
['LEADS_TO', 'TRIGGERS', 'PROVIDES_INPUT_FOR', 'CONVERGES_TO']


**An ACTION node's id must equal a tool name.** That string equality is the entire localization mechanism: no embeddings, no classifier. `tests/test_expert_graphs.py` checks every hand-written graph against the real tool lists.

`neighborhood(node, h)` is a breadth-first search over *both* edge directions, returning the induced subgraph. With `h = 2` from `search` on a seven-node graph it reaches almost everything; the same code keeps the prompt bounded when an evolved graph grows to dozens of nodes. `serialize(active)` is the text form the guidance model receives:

In [3]:
sub = expert.neighborhood("search", h=1)      # h=1 to make the cut visible on a small graph
print(sub.summary(), "around search at h=1  vs ", expert.neighborhood("search", h=2).summary(), "at h=2\n")
show(sub, active="search")
print(sub.serialize(active="search"))

4 nodes, 6 edges around search at h=1  vs  6 nodes, 9 edges at h=2



```mermaid
flowchart LR
  n0(["Start"])
  n1["search"]
  n2{{"analyze"}}
  n3{{"verify"}}
  n0 -->|LEADS_TO| n1
  n1 -->|PROVIDES_INPUT_FOR| n2
  n1 -->|LEADS_TO| n1
  n2 -->|LEADS_TO| n1
  n1 -->|PROVIDES_INPUT_FOR| n3
  n3 -->|LEADS_TO| n1
  classDef active stroke:#2a9d8f,stroke-width:3px
  class n1 active
```

Procedural graph "hotpotqa_expert" (4 nodes, 6 edges)
Active node: [search] (ACTION) Keyword search over the context paragraphs.
Nodes:
  [Start] (STATE) A multi-hop question has been posed.
  [search] (ACTION) Keyword search over the context paragraphs.
  [analyze] (REASONING) Extract the bridge entity or the facts found so far and decide what the second hop needs.
  [verify] (REASONING) Check that both hops are supported by retrieved text and that the answer has the requested form.
Transitions:
  [Start] --LEADS_TO--> [search]
      condition: Always, at the start.
      guidance: First hop: search with the most specific named entity or distinctive phrase from the question, not the whole question.
      pitfalls: Do not answer from memory before retrieving evidence.
  [search] --PROVIDES_INPUT_FOR--> [analyze]
      condition: Search returned paragraphs.
      guidance: Identify the bridge entity (the thing the question refers to indirectly) or, for comparison questions, the attribut

## 3. The environment

A scenario plugs in through two classes in `pg/envs/base.py`: `Environment` (shared, read-only: tasks, system prompt, description, step cap) and `Episode` (all mutable state for one rollout; its tools are closures over it, which is why rollouts can run in a thread pool). `make_env` is the only place a concrete scenario is chosen; nothing in the core imports a scenario.

In [4]:
from pg.envs import make_env

cfg = Config.from_env()            # reads .env; no network yet
env = make_env("hotpotqa", cfg)

print(type(env).__name__, "| max_steps =", env.max_steps)
print({split: len(env.tasks(split)) for split in ("train", "val", "test")})
print("\nsystem prompt the solver gets:\n" + env.system_prompt())

HotpotQAEnv | max_steps = 8
{'train': 100, 'val': 50, 'test': 150}

system prompt the solver gets:
You answer multi-hop questions using tools over a fixed set of context paragraphs.
Use `search` and `lookup` to gather evidence, then call `finish` with a short answer span
(an entity name, a date, a number, or yes/no), not a sentence.


A task: `prompt` is all the solver sees; `meta` is the hidden side, the gold answer and the question's own ten paragraphs (two relevant, eight distractors).

In [5]:
task = env.tasks("train")[0]
print(task.id)
print(task.prompt)
print("gold answer:", task.meta["answer"], "| type:", task.meta["type"], "| level:", task.meta["level"])
for p in task.meta["paragraphs"]:
    print(f"  [{p['title']}] {p['text'][:70]}...")

5a8fab8c5542995b4424208a
Question: Who set Nietzche's philosophical novel to music? 
gold answer: Richard Strauss | type: bridge | level: hard
  [Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach ...
  [Ishmael (novel)] Ishmael is a 1992 philosophical novel by Daniel Quinn. It examines the...
  [The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus....
  [Sidney (novel)] Sidney is a philosophical novel by the American writer Margaret Deland...
  [Theologus Autodidactus] Theologus Autodidactus ("The Self-taught Theologian"), originally titl...
  [The Time of the Angels] The Time of the Angels is a philosophical novel by British novelist Ir...
  [Also sprach Zarathustra (Strauss)] Also sprach Zarathustra , Op. 30 (Thus Spoke Zarathustra or Thus Spake...
  [Marius the Epicurean] Marius the Epicurean: his sensations and ideas is a historical and phi...
  [Hermsprong] Hermsprong: or, Man As He Is Not is a 17

## 4. An episode, driven by hand

`env.start(task)` returns a fresh `Episode`. Here we play the agent ourselves.

In [6]:
episode = env.start(task)
tools = {t.name: t for t in episode.tools}
for t in episode.tools:
    print(f"{t.name}{tuple(t.args)}: {t.description}")

search('query',): Search the context paragraphs. Returns the 3 paragraphs with the highest word overlap with the query.
lookup('title',): Return the full paragraph with the given title.
finish('answer',): Submit the final answer (a short span such as an entity name, a date, or yes/no). Ends the episode.


In [7]:
query = " ".join(task.prompt.replace("Question:", "").split()[:6])     # a deliberately lazy query: the first few words
print("search(%r)\n" % query)
print(tools["search"].invoke({"query": query})[:600])

search("Who set Nietzche's philosophical novel to")

[The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus. First published in 1956, it is his last complete work of fiction. Set in Amsterdam, "The Fall" consists of a series of dramatic monologues by the self-proclaimed "judge-penitent" Jean-Baptiste Clamence, as he reflects upon his life to a stranger. In what amounts to a confession, Clamence tells of his success as a wealthy Parisian defense lawyer who was highly respected by his colleagues; his crisis, and his ultimate "fall" from grace, was meant to invoke, in secular terms, The Fall of Man in the Gar


In [8]:
title = task.meta["paragraphs"][0]["title"]
print(tools["lookup"].invoke({"title": title})[:300])
print("\ndone yet?", episode.is_done())

[Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach Zarathustra: Ein Buch für Alle und Keinen" , also translated as Thus Spake Zarathustra) is a philosophical novel by German philosopher Friedrich Nietzsche, composed in four parts between 1883 and 1885 and 

done yet? False


In [9]:
print(tools["finish"].invoke({"answer": task.meta["answer"]}))   # cheat: submit the gold answer
print("done?", episode.is_done())
episode.score()

Answer submitted.
done? True


{'score': 1.0,
 'success': True,
 'em': 1.0,
 'f1': 1.0,
 'answer': 'Richard Strauss',
 'gold': 'Richard Strauss'}

`score()` is deterministic: SQuAD-style token F1 against the gold answer, `success` = F1 >= 0.5. No model judges anything, here or anywhere else in the loop. A second `env.start(task)` starts from scratch:

In [10]:
fresh = env.start(task)
{t.name: t for t in fresh.tools}["finish"].invoke({"answer": "a wrong answer"})
fresh.score()

{'score': 0.0,
 'success': False,
 'em': 0.0,
 'f1': 0.0,
 'answer': 'a wrong answer',
 'gold': 'Richard Strauss'}

## 5. The solver loop, with a scripted model (offline)

`run_episode` in `agent.py` is the real loop, a four-node LangGraph `StateGraph`:

```
START -> guide -> solver -> tools -> guide ...   (tool call)
                  solver -> text  -> guide ...   (text-only reply; nudge, or accept as final)
```

The `tools` node is hand-written rather than LangGraph's `ToolNode` because the important by-product of acting is the **record**: every call becomes a `Step(index, tool, args, observation)`, which is what the guidance model reads (the last three), what the refiner reads (all of them), and what gets saved. Instead of an LLM we hand it a fake chat model that replays a fixed script of tool calls, the same trick the unit tests use. With `graph=None` the `guide` node is a no-op and this is an ordinary tool-calling agent: the "none" baseline in every comparison.

In [11]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from pg.agent import run_episode

class ScriptedModel(GenericFakeChatModel):
    seen: list = []                                   # every message list the model was invoked with
    def bind_tools(self, tools, **kwargs):
        return self
    def _generate(self, messages, *args, **kwargs):
        self.seen.append(messages)
        return super()._generate(messages, *args, **kwargs)

def call(name, **args):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{name}"}])

script = [call("search", query=query), call("lookup", title=title), call("finish", answer=task.meta["answer"])]
solver = ScriptedModel(messages=iter(script), seen=[])
trajectory = run_episode(env, task, None, cfg, solver=solver)   # graph=None: no guidance

print("score", trajectory.score, "| success", trajectory.success, "| error", trajectory.error)
for s in trajectory.steps:
    print(f"  step {s.index}: {s.tool}({s.args}) -> {s.observation[:70]!r}")

score 1.0 | success True | error None
  step 0: search({'query': "Who set Nietzche's philosophical novel to"}) -> '[The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophic'
  step 1: lookup({'title': 'Thus Spoke Zarathustra'}) -> '[Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and No'
  step 2: finish({'answer': 'Richard Strauss'}) -> 'Answer submitted.'


`compact()` is the one-line-per-step form the refiner reads:

In [12]:
print(trajectory.compact())

task=5a8fab8c5542995b4424208a score=1.00 success=True steps=3
  0. search({"query": "Who set Nietzche's philosophical novel to"}) -> [The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus. First published in 1956, it is his last complete work of fiction. Set in Amsterdam, "The Fall" consists ...
  1. lookup({"title": "Thus Spoke Zarathustra"}) -> [Thus Spoke Zarathustra] Thus Spoke Zarathustra: A Book for All and None (German: "Also sprach Zarathustra: Ein Buch für Alle und Keinen" , also translated as Thus Spake Zarathustra) is a philosophica...
  2. finish({"answer": "Richard Strauss"}) -> Answer submitted.


## 6. Injecting the graph (offline, with the scripted model)

Now the online half. Before every solver turn, `guide` does three things, all in `guidance.py` (43 lines):

1. **Localize**: `match(graph, last_tool)` is the string comparison from section 2.
2. **Cut**: `neighborhood(active, h=2)`, or the whole graph before the first tool call.
3. **Serialize** it, with the task and the last `w = 3` steps, into the guidance model's prompt.

`build_context` is exactly what the guidance model sees. Here it is after our first (lazy) search:

In [13]:
from pg.guidance import build_context

ctx = build_context(expert, task.prompt, trajectory.steps[:1], h=2)
print(ctx[:1600], "\n   [...]")

Task:
Question: Who set Nietzche's philosophical novel to music? 

Recent steps:
0. search({"query": "Who set Nietzche's philosophical novel to"}) -> [The Fall (Camus novel)] The Fall (French: La Chute ) is a philosophical novel by Albert Camus. First published in 1956, it is his last complete work of fiction. Set in Amsterdam, "The Fall" consists of a series of dramatic monologues by the self-proclaimed "judge-penitent" Jean-Baptiste Clamence, as he reflects upon his life to a stranger. In what amounts to a confession, Clamence tells of his success as a wealthy Parisian defense lawyer who was highly respected by his colleagues; his crisis, and his ultimate "fall" from grace, was meant to invoke, in secular terms, The Fall of Man in the Gar

Procedural graph "hotpotqa_expert" (6 nodes, 9 edges)
Active node: [search] (ACTION) Keyword search over the context paragraphs.
Nodes:
  [Start] (STATE) A multi-hop question has been posed.
  [search] (ACTION) Keyword search over the context parag

The guidance model's reply is appended to the **solver's system prompt for that turn only**; it is never a message, so it never enters the history and stale advice cannot pile up. The three lines that do it, in `agent.py`:

```python
system = env.system_prompt()
if state["guidance"]:
    system += "\n\nProcedural Graph Guidance:\n" + state["guidance"]
resp = solver.invoke([SystemMessage(system), *state["messages"]])
```

We can watch it happen without an API key by scripting the guidance model too, and looking at what the solver was actually invoked with:

In [14]:
solver = ScriptedModel(messages=iter(script), seen=[])
guide = ScriptedModel(messages=iter(["Search with the distinctive title phrase, not the whole question.",
                                      "Identify the bridge entity, then look up its paragraph.",
                                      "Both hops are supported: finish with the minimal span."]), seen=[])
guided_traj = run_episode(env, task, expert, cfg, solver=solver, guide_llm=guide)

turn = 1
print("SOLVER SYSTEM PROMPT, turn", turn, ":\n" + solver.seen[turn][0].content)
print("\nrest of the solver's context on that turn:", [type(m).__name__ for m in solver.seen[turn][1:]])
print("guidance in the stored history?", any("Procedural Graph Guidance" in getattr(m, "content", "") for m in solver.seen[turn][1:]))
print("\nguidance model's context on turn", turn, "starts:", guide.seen[turn][1].content.splitlines()[0], "... active node line:",
      next(l for l in guide.seen[turn][1].content.splitlines() if l.startswith("Active node")))
print("\nguidance_log kept on the trajectory:", guided_traj.guidance_log)

SOLVER SYSTEM PROMPT, turn 1 :
You answer multi-hop questions using tools over a fixed set of context paragraphs.
Use `search` and `lookup` to gather evidence, then call `finish` with a short answer span
(an entity name, a date, a number, or yes/no), not a sentence.

Procedural Graph Guidance:
Identify the bridge entity, then look up its paragraph.

rest of the solver's context on that turn: ['HumanMessage', 'AIMessage', 'ToolMessage']
guidance in the stored history? False

guidance model's context on turn 1 starts: Task: ... active node line: Active node: [search] (ACTION) Keyword search over the context paragraphs.

guidance_log kept on the trajectory: ['Search with the distinctive title phrase, not the whole question.', 'Identify the bridge entity, then look up its paragraph.', 'Both hops are supported: finish with the minimal span.']


## 7. A real batch (live)

Six training questions with the real solver and **no graph**. `run_batch` is a thread pool over `run_episode`; this is exactly the call `evolve` makes at the start of every round. Expect about half a cent.

In [15]:
from pg.agent import run_batch

batch = env.tasks("train")[:6]
trajectories = run_batch(env, batch, None, cfg)

    5a808f96554299485f598655: score=0.50 steps=2


    5ab680d15542995eadef0011: score=1.00 steps=2


    5ade66e1554299728e26c713: score=1.00 steps=2


    5a8fab8c5542995b4424208a: score=1.00 steps=3


    5ae3dc205542992f92d823a1: score=1.00 steps=2


    5a803d695542996402f6a494: score=0.47 steps=5


In [16]:
cost = lambda t: sum(v for k, v in t.usage.items() if k.endswith("cost"))
for t in trajectories:
    gold = next(x.meta["answer"] for x in batch if x.id == t.task_id)
    print(f"{t.task_id}  score={t.score:.2f}  steps={len(t.steps)}  ${cost(t):.5f}  answered={t.metrics.get('answer')!r}  gold={gold!r}")

5a8fab8c5542995b4424208a  score=1.00  steps=3  $0.00059  answered='Richard Strauss'  gold='Richard Strauss'
5ade66e1554299728e26c713  score=1.00  steps=2  $0.00035  answered='1970s and 1980s'  gold='1970s and 1980s'
5ab680d15542995eadef0011  score=1.00  steps=2  $0.00022  answered='Pleiospilos'  gold='Pleiospilos'
5a808f96554299485f598655  score=0.50  steps=2  $0.00025  answered='the First Balkan War'  gold='the Slavic women accompanying their husbands in the First Balkan War.'
5a803d695542996402f6a494  score=0.47  steps=5  $0.00089  answered='Plateau is a two-player abstract strategy game, while Colt Express is a railway-themed family game.'  gold='Colt Express is a railway-themed'
5ae3dc205542992f92d823a1  score=1.00  steps=2  $0.00024  answered='minor basilica'  gold='a minor basilica'


## 8. Rank them, as `evolve` does

Contrast is the refiner's signal: it gets the best and the worst attempts and is asked what the winners did that the losers didn't. The ranking is a sort on the environment's own score with errored episodes dropped; the top and bottom `k` (`Config.refiner_k = 3`, or half the batch) become the evidence. With a batch this small and a task this easy expect ties; the sort is stable, so ties keep batch order.

In [17]:
ranked = sorted((t for t in trajectories if not t.error), key=lambda t: t.score, reverse=True)
n = max(1, min(cfg.refiner_k, len(ranked) // 2))
best, worst = ranked[:n], ranked[-n:]
print("best :", [(t.task_id[-6:], round(t.score, 2)) for t in best])
print("worst:", [(t.task_id[-6:], round(t.score, 2)) for t in worst])
print("\n--- the lowest-scoring trajectory, as the refiner would read it ---\n")
print(worst[-1].compact())

best : [('24208a', 1.0), ('26c713', 1.0), ('ef0011', 1.0)]
worst: [('d823a1', 1.0), ('598655', 0.5), ('f6a494', 0.47)]

--- the lowest-scoring trajectory, as the refiner would read it ---

task=5a803d695542996402f6a494 score=0.47 success=False steps=5
  0. search({"query": "Plateau game main difference Colt Express game rules gameplay objective"}) -> [GetAmped2] GetAmped2 (ゲットアンプド2 , GettoAnpudo 2 ) is a free to play massively multiplayer online Fighting-Action, Role-playing video game, developed and published by the Japanese company CyberStep, In...
  1. search({"query": "\"Plateau\" \"Colt Express\" board game"}) -> [Colt Express] Colt Express is a railway-themed family board game designed by Christophe Raimbault, Illustrated by Ian Parovel and Jordi Valbuena, published in 2014 by Ludonaute and distributed by Asm...
  2. lookup({"title": "[Plateau (game)]"}) -> [Plateau (game)] Plateau is a two-player abstract strategy board game invented by Jim Albea.
  3. lookup({"title": "[Colt Ex

## 9. The same batch with the expert graph (live)

One extra LLM call per step. The `Trajectory` keeps every piece of advice in `guidance_log` and accounts for its tokens separately from the solver's.

In [18]:
guided = run_batch(env, batch, expert, cfg)
g = guided[0]
print(batch[0].prompt, "\n")
for i, advice in enumerate(g.guidance_log):
    step = g.steps[i] if i < len(g.steps) else None
    print(f"guidance before step {i}: {advice}")
    if step:
        print(f"   -> {step.tool}({step.args})\n")

    5ade66e1554299728e26c713: score=1.00 steps=2


    5a8fab8c5542995b4424208a: score=1.00 steps=3


    5a808f96554299485f598655: score=0.50 steps=3


    5ae3dc205542992f92d823a1: score=1.00 steps=3


    5ab680d15542995eadef0011: score=1.00 steps=5


    5a803d695542996402f6a494: score=0.44 steps=7


Question: Who set Nietzche's philosophical novel to music?  

guidance before step 0: Begin with the **search** action using the distinctive phrase **“Nietzsche philosophical novel”** (also try the misspelling “Nietzche” if needed), rather than searching the whole question. Look for the paragraph identifying the novel and its composer, and avoid answering from memory before retrieving evidence.
   -> search({'query': 'Nietzsche philosophical novel'})

guidance before step 1: The search identifies **Thus Spoke Zarathustra** as Nietzsche’s philosophical novel. Proceed to **analyze**, then search specifically for **“Thus Spoke Zarathustra” + “set to music”** or **“Also sprach Zarathustra” composer** to establish who composed its musical setting; avoid answering from memory without that second-hop evidence.
   -> search({'query': 'Thus Spoke Zarathustra set to music composer Also sprach Zarathustra'})

guidance before step 2: The search results provide both hops: Nietzsche’s philosophical 

In [19]:
from pg.trajectory import summarize

for name, ts in (("no graph", trajectories), ("expert graph", guided)):
    s = summarize(ts)
    tokens = lambda p: s.get(f"{p}_input_tokens", 0) + s.get(f"{p}_output_tokens", 0)
    print(f"{name:13} mean F1 {s['mean_score']:.3f} | success {s['success_rate']:.0%} | steps {s['mean_steps']:.1f} | "
          f"solver tokens {tokens('solver'):,} | guidance tokens {tokens('guidance'):,} | ${s.get('solver_cost', 0) + s.get('guidance_cost', 0):.4f}")

no graph      mean F1 0.828 | success 83% | steps 2.7 | solver tokens 9,317 | guidance tokens 0 | $0.0025
expert graph  mean F1 0.824 | success 83% | steps 3.8 | solver tokens 17,076 | guidance tokens 27,158 | $0.0125


Six questions prove nothing about which is better; `analysis/paired_comparison.py` and the README's "Measuring whether a graph helps" cover that. The point here is the mechanics: same tasks, same loop, one extra call per step, and guidance typically costs more tokens than solving.

## 10. Edits are data, not code (offline)

The refiner never writes a graph. It fills in an `EditSet`: nodes and edges to add, ids and keys to delete, and a rationale. `apply()` turns that into a **new** graph and leaves the original untouched (deletes run before adds; re-adding an existing `(src, rel, dst)` key revises its text; an edge to an unknown node is dropped with a warning, because LLMs do propose those). `diff()` reports what changed. Here we play the refiner by hand, starting from the scratch skeleton: `Start`, `End`, one ACTION node per tool, **zero edges**.

In [20]:
from pg.graph import EditSet, EdgeRef
from analysis.graph_evolution import text_diff

skeleton = ProceduralGraph.skeleton("hotpotqa_demo", env.tool_descriptions())
print(skeleton.summary(), "->", [n.id for n in skeleton.nodes], "\n")

edits = EditSet(
    rationale="Failures pasted the whole question into search and answered in full sentences.",
    add_nodes=[Node(id="AnswerSpan", type=NodeType.REASONING, description="Reduce the answer to the minimal span.")],
    add_edges=[
        Edge(src="Start", rel=Relation.LEADS_TO, dst="search",
             guidance="Search with the most distinctive entity or phrase, not the whole question."),
        Edge(src="search", rel=Relation.LEADS_TO, dst="AnswerSpan", condition="The answer is present in retrieved text."),
        Edge(src="AnswerSpan", rel=Relation.TRIGGERS, dst="finish", pitfalls="Never submit a sentence; extra words lower F1."),
        Edge(src="search", rel=Relation.LEADS_TO, dst="web_search"),      # no such tool: will be dropped
    ],
)
candidate, warnings = skeleton.apply(edits)
print("warnings:", warnings)
print(candidate.summary(), "\n")
print(text_diff(skeleton.diff(candidate)))
print("\noriginal untouched:", skeleton.summary())

5 nodes, 0 edges -> ['Start', 'End', 'search', 'lookup', 'finish'] 

warnings: ['add_edge: unknown endpoint in search-LEADS_TO->web_search; skipped']
6 nodes, 3 edges 

+ node [AnswerSpan] (REASONING) Reduce the answer to the minimal span.
+ edge Start --LEADS_TO--> search
    guidance: Search with the most distinctive entity or phrase, not the whole question.
+ edge search --LEADS_TO--> AnswerSpan
    condition: The answer is present in retrieved text.
+ edge AnswerSpan --TRIGGERS--> finish
    pitfalls: Never submit a sentence; extra words lower F1.

original untouched: 5 nodes, 0 edges


## 11. What the refiner is shown (offline)

`build_prompt` assembles: the environment description, the tools, the whole current graph as JSON, the best and worst trajectories in compact form, and the rejection memory (edits already tried and rejected). The system prompt (`REFINER_SYSTEM` in `refiner.py`) explains the graph semantics, caps a proposal at about eight operations, and insists on rules that transfer to *any* task: no entity names or question wording copied from the trajectories. We build the exact prompt `evolve` would send for our six trajectories and the skeleton, but do not send it.

In [21]:
from pg.refiner import build_prompt, REFINER_SYSTEM

prompt = build_prompt(env.description, env.tool_descriptions(), skeleton, best, worst, rejected=[],
                      max_chars=cfg.refiner_max_chars, compact=env.compact_trajectory)
print(f"{len(prompt):,} characters; sections:", [l for l in prompt.splitlines() if l.startswith("## ")])
print()
print(prompt[-1500:])

5,094 characters; sections: ['## Environment', '## Tools', '## Current graph (JSON)', '## Highest-scoring trajectories', '## Lowest-scoring trajectories', '## Rejection memory (edits already tried and rejected)']

n (Russian: Васи́лий Ива́нович Ага́пкин ; 3 February 1884 – 29 October 1964) was a Soviet military orchestra conductor, composer, and author of the well-known m...
  1. finish({"answer": "the First Balkan War"}) -> Answer submitted.

task=5a803d695542996402f6a494 score=0.47 success=False steps=5
  0. search({"query": "Plateau game main difference Colt Express game rules gameplay objective"}) -> [GetAmped2] GetAmped2 (ゲットアンプド2 , GettoAnpudo 2 ) is a free to play massively multiplayer online Fighting-Action, Role-playing video game, developed and published by the Japanese company CyberStep, In...
  1. search({"query": "\"Plateau\" \"Colt Express\" board game"}) -> [Colt Express] Colt Express is a railway-themed family board game designed by Christophe Raimbault, Illustrated by 

## 12. A real evolution (live: three rounds, about $1)

Everything above, in a loop, with the real refiner and the validation gate. Per round `evolve` (a) rolls out the current graph on the next train batch, (b) ranks and prompts the refiner, (c) applies the `EditSet` to a copy, (d) runs the candidate on a **fixed, held-out validation split** with the full online machinery, and (e) keeps it only if its mean score is at least the current one; a rejected proposal is summarised into the next prompt's rejection memory. Nothing an LLM says is trusted: the only test is behavioural, on tasks the refiner never saw.

Three rounds with a batch of 6 and 12 validation questions keep this to about a dollar and 10–15 minutes; a real run would use the whole val split (`--val-n` default) and more rounds.

In [22]:
from pg.evolve import evolve

out = ROOT / "graphs" / "evolved" / "walkthrough-demo"
if (out / "evolution_log.jsonl").exists():          # re-opening the notebook: reuse the run instead of paying again
    best_graph = ProceduralGraph.load(out / "best.json")
    print("reusing the existing run in", out.relative_to(ROOT), "(delete it to evolve afresh)")
else:
    best_graph = evolve(env, ProceduralGraph.skeleton("hotpotqa_walkthrough", env.tool_descriptions()),
                        cfg, rounds=3, batch=6, val_n=12, out_dir=out)
print("\nfinal graph:", best_graph.summary())


reusing the existing run in graphs/evolved/walkthrough-demo (delete it to evolve afresh)

final graph: 7 nodes, 9 edges


The log has one entry per round: batch score, validation before and after, the verdict, the edit summary and the refiner's rationale. Everything else a round produced is on disk too: `round_k.json` (graph kept), `edits/round_k.json` (what was proposed, accepted or not), `prompts/round_k.txt`, and every trajectory.

In [23]:
import json
log = [json.loads(l) for l in (out / "evolution_log.jsonl").read_text().splitlines()]
for e in log:
    after = e.get("val_score", e.get("val_after"))
    verdict = "baseline" if e["round"] == 0 else ("ACCEPTED" if e["accepted"] else "rejected" if after is not None else "not validated")
    print(f"round {e['round']}: val {e.get('val_before', after):.3f} -> {after if after is None else round(after, 3)}  {verdict:12} {e['graph']}")
    if e.get("rationale"):
        print("   rationale:", e["rationale"][:300].replace("\n", " "), "...")
    if e.get("edits"):
        print("   edits:", e["edits"][:200])
print("\nfiles:", sorted(p.name for p in out.iterdir()))

round 0: val 0.727 -> 0.727  baseline     5 nodes, 0 edges
round 1: val 0.727 -> 0.821  ACCEPTED     6 nodes, 7 edges
   rationale: The graph had no edges, so nothing constrained the agent. The trajectories show three recurring procedural facts. (1) The winning runs always treated the first search as retrieval-only and, when the top hits were topical distractors, reformulated the query with the concrete bridge entity/attribute t ...
   edits: add nodes: VerifyEvidence; add/revise edges: Start-LEADS_TO->search, search-LEADS_TO->VerifyEvidence, VerifyEvidence-TRIGGERS->search, VerifyEvidence-TRIGGERS->lookup, lookup-LEADS_TO->VerifyEvidence,
round 2: val 0.821 -> 0.825  ACCEPTED     7 nodes, 9 edges
   rationale: The failing trajectory is a 'what do A and B share' style question: the agent correctly retrieved each entity's own paragraph within 3 steps, then spent 5 more steps on redundant lookups of paragraphs it had already read in full and on queries naming both entities at once, hopin

A rejected round is summarised into the rejection memory of every *later* prompt, with the instruction not to re-propose it. With only three rounds the rejection may be the last one, in which case no later prompt exists yet; the cell shows the entry that would be carried, and what the saved prompts actually contained.


In [24]:
rejected = [e for e in log[1:] if e["accepted"] is False and e["val_after"] is not None]
for e in rejected:
    print(f"round {e['round']} was rejected (val {e['val_before']:.3f} -> {e['val_after']:.3f}); this is what a later prompt's rejection memory would carry:")
    print(f"  - round {e['round']}: {e['edits'][:160]}... | val {e['val_before']:.3f} -> {e['val_after']:.3f} | rationale: {e['rationale'][:120]}...\n")
for k in range(2, len(log)):
    text = (out / "prompts" / f"round_{k}.txt").read_text()
    mem = text[text.index("## Rejection memory"):].split("\n\nPropose")[0]
    print(f"--- as saved in the prompt for round {k} ---\n{mem[:600]}\n")


round 3 was rejected (val 0.825 -> 0.814); this is what a later prompt's rejection memory would carry:
  - round 3: add nodes: RetrievalStalled; add/revise edges: search-LEADS_TO->RetrievalStalled, RetrievalStalled-CONVERGES_TO->finish, RetrievalStalled-TRIGGERS->lookup, Veri... | val 0.825 -> 0.814 | rationale: The failures cluster into three procedural defects that the successes avoid. (a) One failure submitted an answer after a...

--- as saved in the prompt for round 2 ---
## Rejection memory (edits already tried and rejected)
(empty)

--- as saved in the prompt for round 3 ---
## Rejection memory (edits already tried and rejected)
(empty)



## 13. Reading what evolution did

`analysis/graph_evolution.py` turns a run directory into a text diff per round, a Markdown report with one Mermaid diagram per round, and a self-contained interactive timeline (fixed layout across rounds; click an edge to read its text, or the text it replaced).

In [25]:
from analysis.graph_evolution import load_frames, report, to_mermaid

frames = load_frames(out)
last = next(f for f in reversed(frames) if f["accepted"])      # the last accepted round
print(f"round {last['round']} changes:\n{last['changes'][:1500]}")

round 2 changes:
+ node [SynthesizeAcrossParagraphs] (REASONING) Derive the answer by comparing the two retrieved paragraphs instead of retrieving more: list each entity's attributes (type, nationality/origin, era, genre, role, quantity) and name the single attribute they share or on which they differ. Cross-entity relations are almost never stated in any one paragraph; they must be inferred from the two.
+ edge VerifyEvidence --TRIGGERS--> SynthesizeAcrossParagraphs
    condition: The question involves two entities (what they have in common, which one is X-er, same/different category, yes/no about both) and each entity's own paragraph has already been returned by search or lookup.
    guidance: Stop retrieving. Put the two paragraphs side by side and compare the attribute class the question names (origin/country, entity type, genre, date, count, occupation). The answer is the shared or distinguishing attribute phrased with the wording used in the paragraphs.
    pitfalls: Do not issue

In [26]:
from IPython.display import Markdown, display
display(Markdown("```mermaid\n" + to_mermaid(last) + "\n```"))       # renders in JupyterLab 4 / VS Code; else copy into a Mermaid viewer
md_path, html_path = report(out)
print("wrote", md_path, "and", html_path, "(open the HTML in a browser)")

```mermaid
flowchart LR
  n0(["Start"])
  n1(["End"])
  n2["search"]
  n3["lookup"]
  n4["finish"]
  n5{{"VerifyEvidence"}}
  n6{{"SynthesizeAcrossParagraphs"}}
  n0 -->|LEADS_TO| n2
  n2 -->|LEADS_TO| n5
  n5 -->|TRIGGERS| n2
  n5 -->|TRIGGERS| n3
  n3 -->|LEADS_TO| n5
  n5 -->|CONVERGES_TO| n4
  n4 -->|LEADS_TO| n1
  n5 -->|TRIGGERS| n6
  n6 -->|CONVERGES_TO| n4
  linkStyle 2 stroke:#e9a23b,stroke-width:3px
  linkStyle 3 stroke:#e9a23b,stroke-width:3px
  linkStyle 4 stroke:#e9a23b,stroke-width:3px
  linkStyle 7 stroke:#2a9d8f,stroke-width:3px
  linkStyle 8 stroke:#2a9d8f,stroke-width:3px
  classDef added stroke:#2a9d8f,stroke-width:3px
  class n6 added
```

wrote /Users/paul/code/procedural-graphs/graphs/evolved/walkthrough-demo/evolution.md and /Users/paul/code/procedural-graphs/graphs/evolved/walkthrough-demo/timeline.html (open the HTML in a browser)


## Where this leaves you

- The **online half** is a string match, a two-hop BFS, and one paragraph appended to the system prompt for a single turn.
- The **offline half** is a loop that shows a strong model the best and worst attempts, takes back a typed list of edits, and keeps them only if a held-out set agrees, remembering what did not work.
- The agent's know-how ends up as an **artifact**: a JSON file you can diff between rounds, review, roll back, and hand to a colleague, with a written rationale and a validation score for every edit.

Whether a graph *helps* is a separate, empirical question with a separate answer per task: on HotpotQA it did not (the base model already solves these about as well as the tools allow), on the paper's finance benchmark a graph evolved from scratch took survival from 1/20 to 17/19. Those results, the controls that separate "the graph's content helps" from "any extra LLM call helps", and where to change things are in [`walkthrough.md`](walkthrough.md).